In [1]:
using Falcons
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")
using Base.Threads
using Statistics
using NPZ

In [2]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(10.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = 2)
;
cs = ConvolutionSky(lmax = lmax_in, numofsky = 2)
@show fieldnames(typeof(cs))
cb = ConvolutionBeam(lmax = lmax_in, mmax =2, numofbeams = 2)
@show fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0,)
@show fieldnames(typeof(cc))

fieldnames(typeof(cs)) = (:numofsky, :lmax, :alm)
fieldnames(typeof(cb)) = (:numofbeams, :lmax, :mmax, :blm)
fieldnames(typeof(cc)) = (:nside_output, :resol, :lstart, :lstop, :mmax_calculate, :HWP)


(:nside_output, :resol, :lstart, :lstop, :mmax_calculate, :HWP)

In [3]:
alm_in = npzread("./inputs/alm=CMB=r0=nside32=seed200.npy")
alm_in[1,:] .= 0
cs.alm[1,:,:] = alm_in
cs.alm[2,:,:] = alm_in

3×4656 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im          0.0+0.0im  …         0.0+0.0im
 0.0+0.0im  0.0+0.0im    -0.335517+0.0im      0.0102629+0.0115285im
 0.0+0.0im  0.0+0.0im  -0.00174782+0.0im     0.00121166+0.00034225im

In [4]:
cb.blm[:,:,1] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
cb.blm[:,:,2] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)


285×3 Matrix{ComplexF64}:
 0.282095+0.0im           0.0+0.0im  0.0+0.0im
 0.485926+0.0im           0.0+0.0im  0.0+0.0im
 0.620473+0.0im           0.0+0.0im  0.0+0.0im
 0.722154+0.0im           0.0+0.0im  0.0+0.0im
 0.801049+0.0im           0.0+0.0im  0.0+0.0im
 0.861599+0.0im           0.0+0.0im  0.0+0.0im
 0.906288+0.0im           0.0+0.0im  0.0+0.0im
 0.936785+0.0im           0.0+0.0im  0.0+0.0im
 0.954405+0.0im           0.0+0.0im  0.0+0.0im
 0.960314+0.0im           0.0+0.0im  0.0+0.0im
 0.955628+0.0im           0.0+0.0im  0.0+0.0im
 0.941456+0.0im           0.0+0.0im  0.0+0.0im
 0.918919+0.0im           0.0+0.0im  0.0+0.0im
         ⋮                           
      0.0+0.0im    -5.5136e-9+0.0im  0.0-5.5136e-9im
      0.0+0.0im   -3.47698e-9+0.0im  0.0-3.47698e-9im
      0.0+0.0im   -2.18049e-9+0.0im  0.0-2.18049e-9im
      0.0+0.0im   -1.35985e-9+0.0im  0.0-1.35985e-9im
      0.0+0.0im  -8.43358e-10+0.0im  0.0-8.43358e-10im
      0.0+0.0im  -5.20141e-10+0.0im  0.0-5.20141e-10im


In [5]:
alm_slice = slice_spin_alm_by_l(cs, cc)
blm_slice = slice_spin_blm_by_l(cb, cc);

In [6]:
ss = gen_ScanningStrategy(nside = 32,coord="G")
show_ss(ss)

nside                    : 32 
duration [sec]           : 31536000.0 
sampling rate [Hz]       : 1.0 
alpha [deg]              : 45.0 
beta [deg]               : 50.0 
gamma [deg]              : 0.0 
prec. period [min]       : 192.348
↳ prec. rate [rpm]       : 0.005199
spin period [min]        : 20.000
↳ spin rate [rpm]        : 0.050000
HWP rot. rate[rpm]       : 0.000000 
start angle              : 0.000000 
coordinate system        : G 
FPU                     
↳ Det. 1  name | boresight              : (x,y,z,w) = [0.000, 0.000, 1.000, 0.000] 


In [7]:
nptg = 31536000
pointings_ = zeros(nptg,3)

pointings_[:,1], pointings_[:,2], pointings_[:,3], time_array = get_pointings(ss, 0, nptg)
;

In [8]:
pixels = ang2pixRing.(Ref(cc.resol), pointings_[:,1], pointings_[:,2]);

In [9]:
index_dict = build_index_dict(pixels)


Dict{Int64, Vector{Int64}} with 12288 entries:
  11950 => [9744, 67341, 67342, 67343, 67344, 67345, 85056, 85057, 85058, 85059…
  1703  => [5421005, 5421006, 5421007, 5455784, 5455785, 5455786, 5455787, 5455…
  7685  => [41590, 41591, 90397, 90398, 90399, 90400, 90401, 90402, 90403, 9040…
  3406  => [14147405, 14147406, 14147407, 14147408, 14182183, 14182184, 1418218…
  2015  => [8539799, 8539800, 8597412, 8597413, 8597414, 8632191, 8632192, 8632…
  1090  => [8687394, 8687395, 8687396, 8687397, 8687398, 8687399, 8687400, 8710…
  11280 => [13016, 13017, 13018, 13019, 13020, 13021, 40985, 40986, 40987, 1054…
  11251 => [19500, 19501, 19502, 19503, 19504, 19505, 19506, 27296, 27297, 2729…
  3220  => [3931816, 3931817, 3931818, 3931819, 3931820, 3931821, 3931822, 3966…
  422   => [9496195, 9496196, 9496197, 9519029, 9519030, 9530974, 9530975, 9530…
  4030  => [13061391, 13061392, 13061393, 13061394, 13061395, 13084223, 1308422…
  8060  => [33900, 33901, 33902, 33903, 33904, 33905, 33906, 3

In [ ]:
nring = 4*cc.resol.nside - 1
for i in 1:nring
    start, stop = ring_pixel_range(i, cc.resol.nside)
    temp_theta = unique_theta_val(i, cc.resol.nside)
    @time temp_gd = global_wignerd(cc, temp_theta)
    println("Ring $i: pixels $start to $stop")
    for pix in start:stop
        indices = get(index_dict, pix, Int[])
        ang = pix2angRing(cc.resol, pix)
        @time pix_D = global_d2D_conj(cc, temp_gd, ang[2])
        φ = pointings_[indices,2]
        θ = pointings_[indices,1]
        ψ = pointings_[indices,3]
        #@show length(indices)
        @time test_map = compute_pixel_convolution_mapmake(cs, cb, cc, alm_slice, blm_slice, pix, pix_D, φ, θ, ψ; τ=10)
    end
end

  7.716561 seconds (1.30 M allocations: 242.337 MiB, 7.56% compilation time)
Ring 1: pixels 1 to 4
  3.896566 seconds (79.34 k allocations: 62.210 MiB, 1.41% compilation time)
 18.230816 seconds (5.80 M allocations: 412.135 MiB, 0.21% gc time, 8.14% compilation time)
  3.755558 seconds (34 allocations: 56.785 MiB)
 16.291704 seconds (39.15 k allocations: 21.687 MiB)
  3.753311 seconds (34 allocations: 56.785 MiB)
 16.352852 seconds (39.74 k allocations: 21.726 MiB)
  3.873580 seconds (34 allocations: 56.785 MiB)
 16.559147 seconds (39.94 k allocations: 21.739 MiB)
  5.128888 seconds (33 allocations: 56.785 MiB)
Ring 2: pixels 5 to 12
  3.760260 seconds (34 allocations: 56.785 MiB)
 16.259223 seconds (39.99 k allocations: 21.743 MiB)
  4.095751 seconds (34 allocations: 56.785 MiB, 5.67% gc time)
 16.446289 seconds (39.07 k allocations: 21.683 MiB)
